# Interactive Privacy Parameter Tuning for Media Mix Modeling

This notebook provides an interactive exploration of how the privacy parameter (epsilon, ε) affects both privacy guarantees and model accuracy in differential privacy for Media Mix Modeling.

## Learning Objectives

1. **Understand the privacy-utility tradeoff**: See how epsilon affects privacy and accuracy
2. **Visualize noise impact**: Observe how differential privacy adds calibrated noise to data
3. **Compare model performance**: Evaluate models across different epsilon values
4. **Make informed decisions**: Learn to select appropriate epsilon for your use case

## What is Epsilon (ε)?

Epsilon is the **privacy budget** in differential privacy:
- **Lower epsilon** = More privacy, more noise, less accuracy
- **Higher epsilon** = Less privacy, less noise, more accuracy

Typical ranges:
- ε ≤ 1.0: High privacy (sensitive health/financial data)
- ε = 1.0-3.0: Moderate privacy (most marketing applications)
- ε ≥ 5.0: Low privacy (less sensitive aggregated data)

## Setup and Imports

In [ ]:
# Enable interactive matplotlib for better widget integration
%matplotlib inline

import sys
import os

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Import MMM functions
from advanced_mmm import (
    generate_weekly_data,
    apply_differential_privacy,
    fit_model,
    CONFIG,
    geometric_adstock,
    hill_function
)

print("✓ Imports successful!")
print(f"Number of marketing channels: {len(CONFIG['channels'])}")
print(f"Channels: {', '.join(CONFIG['channels'])}")

## Generate Synthetic Data

We'll generate one set of synthetic data to use throughout this notebook. This ensures fair comparisons between different epsilon values.

In [ ]:
# Generate synthetic data
print("Generating synthetic weekly data...")
base_data = generate_weekly_data()
print(f"Generated {len(base_data)} weeks of data")

# Display first few rows
display(base_data.head())

# Visualize the "true" revenue pattern (without privacy noise)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Revenue over time
axes[0, 0].plot(base_data['week'], base_data['revenue'], linewidth=1.5)
axes[0, 0].set_title('Original Revenue (No Privacy Noise)', fontweight='bold')
axes[0, 0].set_xlabel('Week')
axes[0, 0].set_ylabel('Revenue')
axes[0, 0].grid(True, alpha=0.3)

# Spend distributions
spend_data = [base_data[f'spend_{ch}'] for ch in CONFIG['channels']]
axes[0, 1].boxplot(spend_data, labels=CONFIG['channels'])
axes[0, 1].set_title('Marketing Spend Distribution by Channel', fontweight='bold')
axes[0, 1].set_ylabel('Spend')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Revenue histogram
axes[1, 0].hist(base_data['revenue'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Revenue Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Revenue')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Contributions by channel
for ch in CONFIG['channels']:
    if f'contribution_{ch}' in base_data.columns:
        axes[1, 1].plot(base_data['week'], base_data[f'contribution_{ch}'], 
                       label=ch, linewidth=1.5, alpha=0.7)
axes[1, 1].set_title('True Channel Contributions', fontweight='bold')
axes[1, 1].set_xlabel('Week')
axes[1, 1].set_ylabel('Contribution to Revenue')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Data generated and visualized!")

## Interactive Exploration: Noise Visualization

Use the slider below to adjust epsilon and see how it affects the amount of noise added to the data. This is your first hands-on experience with the privacy-utility tradeoff!

In [ ]:
def visualize_privacy_noise(epsilon):
    """
    Visualize the impact of epsilon on data noise.
    """
    # Apply differential privacy
    private_data = apply_differential_privacy(base_data.copy(), epsilon)
    
    # Create visualization
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Plot 1: Revenue comparison
    axes[0].plot(base_data['week'], base_data['revenue'], 
                label='Original', linewidth=2, alpha=0.7)
    axes[0].plot(private_data['week'], private_data['revenue'], 
                label=f'With Privacy (ε={epsilon})', linewidth=2, alpha=0.7)
    axes[0].set_title('Revenue: Original vs Private', fontweight='bold')
    axes[0].set_xlabel('Week')
    axes[0].set_ylabel('Revenue')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Noise distribution
    revenue_noise = private_data['revenue'] - base_data['revenue']
    axes[1].hist(revenue_noise, bins=30, edgecolor='black', alpha=0.7, color='coral')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=2, alpha=0.5)
    axes[1].set_title(f'Noise Distribution (ε={epsilon})', fontweight='bold')
    axes[1].set_xlabel('Noise Added to Revenue')
    axes[1].set_ylabel('Frequency')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Noise statistics
    noise_stats = []
    noise_labels = []
    for ch in CONFIG['channels']:
        spend_col = f'spend_{ch}'
        noise = (private_data[spend_col] - base_data[spend_col]).abs()
        noise_stats.append(noise.mean())
        noise_labels.append(f'{ch}\nSpend')
    
    revenue_noise_mean = revenue_noise.abs().mean()
    noise_stats.append(revenue_noise_mean)
    noise_labels.append('Revenue')
    
    axes[2].bar(range(len(noise_stats)), noise_stats, color='steelblue', alpha=0.7)
    axes[2].set_xticks(range(len(noise_stats)))
    axes[2].set_xticklabels(noise_labels, rotation=0)
    axes[2].set_title('Mean Absolute Noise by Metric', fontweight='bold')
    axes[2].set_ylabel('Mean Absolute Noise')
    axes[2].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    # Privacy guarantee description
    if epsilon <= 0.5:
        privacy_level = "Very High Privacy"
        description = "Excellent privacy protection, suitable for highly sensitive data."
    elif epsilon <= 1.0:
        privacy_level = "High Privacy"
        description = "Strong privacy guarantees, recommended for sensitive marketing data."
    elif epsilon <= 3.0:
        privacy_level = "Moderate Privacy"
        description = "Balanced privacy-utility tradeoff for most marketing applications."
    elif epsilon <= 5.0:
        privacy_level = "Low Privacy"
        description = "Minimal privacy protection, higher model accuracy."
    else:
        privacy_level = "Very Low Privacy"
        description = "Very little privacy protection, focus on utility."
    
    print(f"\n🔒 Privacy Level: {privacy_level}")
    print(f"   {description}")
    print(f"\n📊 Noise Statistics (ε={epsilon}):")
    print(f"   Mean Revenue Noise: {revenue_noise_mean:.2f}")
    print(f"   Max Revenue Noise: {revenue_noise.abs().max():.2f}")

# Create interactive widget
epsilon_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Epsilon (ε):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

widgets.interact(visualize_privacy_noise, epsilon=epsilon_slider);

## Interactive Model Comparison

Now let's go deeper. Compare two different epsilon values side-by-side to see how they affect the fitted model and predictions.

**Note:** Model fitting can take 10-20 seconds per epsilon value.

In [ ]:
# Cache fitted models to avoid refitting
fitted_models_cache = {}

def calculate_predictions(df, fitted_params):
    """Calculate predicted revenue using fitted parameters."""
    predicted_revenue = CONFIG["base_revenue"] + \
                        CONFIG["true_params"]["seasonality_amplitude"] * \
                        np.sin(2 * np.pi * df['week'] / CONFIG["true_params"]["seasonality_period"])

    predicted_revenue += df["promotions"] * fitted_params["promo_effect"]

    for ch in CONFIG["channels"]:
        params = fitted_params[ch]
        adstocked_spend = geometric_adstock(df[f"spend_{ch}"].values, params["adstock_decay"])
        predicted_revenue += hill_function(
            adstocked_spend,
            params["hill_alpha"],
            params["hill_K"],
            params["hill_beta"]
        )
    return predicted_revenue

def compare_models(epsilon1, epsilon2):
    """
    Compare two models with different epsilon values side-by-side.
    """
    epsilons = [epsilon1, epsilon2]
    results = []
    
    for eps in epsilons:
        # Check cache
        if eps not in fitted_models_cache:
            print(f"Fitting model for ε={eps}... (this may take a moment)")
            private_data = apply_differential_privacy(base_data.copy(), eps)
            fitted_params = fit_model(private_data)
            predictions = calculate_predictions(private_data, fitted_params)
            
            # Calculate R²
            ss_res = np.sum((private_data["revenue"] - predictions) ** 2)
            ss_tot = np.sum((private_data["revenue"] - private_data["revenue"].mean()) ** 2)
            r_squared = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
            
            fitted_models_cache[eps] = {
                'private_data': private_data,
                'fitted_params': fitted_params,
                'predictions': predictions,
                'r_squared': r_squared
            }
        
        results.append(fitted_models_cache[eps])
    
    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = ['#2E86AB', '#A23B72']
    
    # Plot 1: Predicted vs Actual (Model 1)
    axes[0, 0].scatter(results[0]['private_data']['revenue'], results[0]['predictions'], 
                      alpha=0.5, color=colors[0], s=50)
    axes[0, 0].plot([0, results[0]['private_data']['revenue'].max()], 
                   [0, results[0]['private_data']['revenue'].max()], 
                   'k--', linewidth=2, alpha=0.5)
    axes[0, 0].set_title(f'Model 1: ε={epsilon1} (R²={results[0]["r_squared"]:.4f})', fontweight='bold')
    axes[0, 0].set_xlabel('Actual Revenue')
    axes[0, 0].set_ylabel('Predicted Revenue')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Predicted vs Actual (Model 2)
    axes[0, 1].scatter(results[1]['private_data']['revenue'], results[1]['predictions'], 
                      alpha=0.5, color=colors[1], s=50)
    axes[0, 1].plot([0, results[1]['private_data']['revenue'].max()], 
                   [0, results[1]['private_data']['revenue'].max()], 
                   'k--', linewidth=2, alpha=0.5)
    axes[0, 1].set_title(f'Model 2: ε={epsilon2} (R²={results[1]["r_squared"]:.4f})', fontweight='bold')
    axes[0, 1].set_xlabel('Actual Revenue')
    axes[0, 1].set_ylabel('Predicted Revenue')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Parameter comparison
    param_comparison = []
    param_names = []
    for ch in CONFIG['channels']:
        for param_name in ['adstock_decay', 'hill_alpha']:
            true_val = CONFIG['true_params'][ch][param_name]
            fit1_val = results[0]['fitted_params'][ch][param_name]
            fit2_val = results[1]['fitted_params'][ch][param_name]
            
            param_comparison.append([true_val, fit1_val, fit2_val])
            param_names.append(f'{ch}\n{param_name}')
    
    x = np.arange(len(param_names))
    width = 0.25
    
    param_comparison = np.array(param_comparison)
    axes[1, 0].bar(x - width, param_comparison[:, 0], width, label='True', color='green', alpha=0.7)
    axes[1, 0].bar(x, param_comparison[:, 1], width, label=f'ε={epsilon1}', color=colors[0], alpha=0.7)
    axes[1, 0].bar(x + width, param_comparison[:, 2], width, label=f'ε={epsilon2}', color=colors[1], alpha=0.7)
    axes[1, 0].set_title('Parameter Comparison (Selected Parameters)', fontweight='bold')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(param_names, fontsize=8)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Time series comparison
    axes[1, 1].plot(base_data['week'], base_data['revenue'], 
                   label='True Revenue', linewidth=2, alpha=0.5, color='black')
    axes[1, 1].plot(results[0]['private_data']['week'], results[0]['predictions'], 
                   label=f'ε={epsilon1} Predictions', linewidth=1.5, alpha=0.7, color=colors[0])
    axes[1, 1].plot(results[1]['private_data']['week'], results[1]['predictions'], 
                   label=f'ε={epsilon2} Predictions', linewidth=1.5, alpha=0.7, color=colors[1])
    axes[1, 1].set_title('Revenue Predictions Over Time', fontweight='bold')
    axes[1, 1].set_xlabel('Week')
    axes[1, 1].set_ylabel('Revenue')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Comparison Summary:")
    print(f"   Model 1 (ε={epsilon1}): R² = {results[0]['r_squared']:.4f}")
    print(f"   Model 2 (ε={epsilon2}): R² = {results[1]['r_squared']:.4f}")
    
    if results[0]['r_squared'] > results[1]['r_squared']:
        print(f"   ✓ Model 1 has better fit (but less privacy)")
    else:
        print(f"   ✓ Model 2 has better fit (but less privacy)")

# Create comparison widgets
epsilon1_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Epsilon 1:',
    continuous_update=False
)

epsilon2_slider = widgets.FloatSlider(
    value=5.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Epsilon 2:',
    continuous_update=False
)

widgets.interact(compare_models, epsilon1=epsilon1_slider, epsilon2=epsilon2_slider);

## Guided Exercises

Now that you've explored the interactive tools, test your understanding with these exercises:

### Exercise 1: Finding the Privacy Threshold
Go back to the noise visualization widget above. **Adjust epsilon until the noise becomes barely noticeable** in the revenue plot.

**Questions:**
- What epsilon value achieves this?
- Is this level of privacy acceptable for your use case?
- How much noise is being added (check the statistics)?

### Exercise 2: Minimum Viable Utility
Using the model comparison widget, find the **lowest epsilon value (highest privacy)** that still produces acceptable parameter estimates.

**Questions:**
- What R² threshold do you consider "acceptable" for your business?
- What is the minimum epsilon that meets this threshold?
- How much privacy are you sacrificing compared to ε=0.1?

### Exercise 3: Sensitivity Analysis
Compare three epsilon values: 0.5, 2.0, and 8.0 using the comparison widget.

**Questions:**
- How much does R² improve as epsilon increases?
- Are the parameter estimates consistently better with higher epsilon?
- At what point do you see diminishing returns (epsilon increase doesn't help much)?

### Exercise 4: Regulatory Compliance
Research privacy regulations in your jurisdiction (e.g., GDPR, CCPA, HIPAA).

**Questions:**
- What epsilon range would satisfy requirements while maintaining analytical value?
- How would you document and justify your epsilon choice to a privacy officer?
- What additional safeguards might you implement beyond differential privacy?

---

**Record your findings below:**

### My Exercise Notes

**Exercise 1:**
- Chosen epsilon: 
- Reasoning:

**Exercise 2:**
- Minimum acceptable R²:
- Minimum epsilon:
- Reasoning:

**Exercise 3:**
- Observations:

**Exercise 4:**
- Regulatory requirements:
- Recommended epsilon range:
- Justification:

## Best Practices Summary

Based on your exploration and the research behind differential privacy, here are key recommendations:

### Epsilon Selection Guidelines

| Epsilon Range | Privacy Level | Typical Use Cases | Considerations |
|---------------|---------------|-------------------|----------------|
| ε ≤ 0.5 | Very High | Medical records, financial transactions | High noise, may impact utility significantly |
| ε = 0.5-1.0 | High | Sensitive personal data, GDPR compliance | Good balance for regulated industries |
| ε = 1.0-3.0 | Moderate | Marketing analytics, behavioral data | **Recommended starting point** for MMM |
| ε = 3.0-5.0 | Low | Aggregated business metrics | Better utility, acceptable for most marketing |
| ε ≥ 5.0 | Very Low | Public data, non-sensitive analytics | Minimal privacy protection |

### Decision Framework

When selecting epsilon, consider:

1. **Data Sensitivity**
   - Who are the individuals in your data?
   - What could be inferred about them from the analysis?
   - What are the potential harms from privacy breaches?

2. **Regulatory Requirements**
   - GDPR: Typically requires ε ≤ 1.0
   - CCPA: Moderate privacy, ε ≤ 3.0 often acceptable
   - HIPAA: Very strict, ε ≤ 0.5 recommended

3. **Dataset Size**
   - Smaller datasets need higher epsilon for useful results
   - Larger datasets can achieve good utility with lower epsilon
   - Rule of thumb: n > 1000 records allows ε ≤ 1.0

4. **Utility Requirements**
   - High-stakes decisions (budget allocation): Prefer higher epsilon
   - Exploratory analysis: Can tolerate lower epsilon
   - A/B test first if possible

### Documentation Template

Always document your epsilon selection:

```
Privacy Parameter Selection for [Analysis Name]
Date: [Date]
Analyst: [Your Name]

Selected Epsilon: ε = [value]

Justification:
- Data sensitivity: [description]
- Regulatory requirements: [applicable regulations]
- Dataset size: [number of records]
- Utility requirements: [acceptable R², MAE thresholds]
- Testing conducted: [summary of exploration]

Impact Assessment:
- Privacy guarantee: (ε=[value], δ=1e-5)-differential privacy
- Model accuracy: R²=[value], MAE=[value]
- Mean noise level: [value]

Approval: [Privacy officer signature]
```

### Next Steps

1. **Run the parameter sweep script** for comprehensive comparison:
   ```bash
   python examples/privacy_parameter_sweep.py
   ```

2. **Read the decision guide** for deeper analysis:
   - `docs/privacy_parameter_guide.md`

3. **Apply your chosen epsilon** in the main script:
   - Edit `CONFIG["epsilon"]` in `advanced_mmm.py`
   - Document your choice
   - Run the full analysis

4. **Validate results** with stakeholders:
   - Present privacy-utility tradeoff visualizations
   - Explain the differential privacy guarantee
   - Obtain sign-off from privacy officer

---

## Conclusion

You've now explored the privacy-utility tradeoff hands-on! Remember:

- **There's no one-size-fits-all epsilon** - it depends on your specific context
- **Lower epsilon = more privacy** but also more noise and lower accuracy
- **Document your decision** with clear justification
- **Test multiple values** before committing to production

Differential privacy is a powerful tool for privacy-preserving analytics. By understanding and carefully tuning epsilon, you can achieve meaningful insights while respecting individual privacy.

**Happy modeling!** 🔒📊